In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import wfdb
import ast
import torch
import seaborn as sns
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, confusion_matrix
from tabulate import tabulate
import os
from collections import Counter
from sklearn.preprocessing import StandardScaler

In [2]:
import scipy.signal as signal
def filter_signal(signal_data, axis=0, fs=100, lowcut=0.5, highcut=45.0):
        nyquist = 0.5 * fs
        low = lowcut / nyquist
        high = highcut / nyquist
        b, a = signal.butter(4, [low, high], btype='band')
        return signal.filtfilt(b, a, signal_data, axis=axis)

In [3]:
def normalize_data_z_score(data: np.ndarray) -> np.ndarray:
    num_samples, sample_length, num_channels = data.shape
    data_reshaped = data.reshape(-1, num_channels)
    scaler = StandardScaler()
    data_normalized = scaler.fit_transform(data_reshaped)
    return data_normalized.reshape(num_samples, sample_length, num_channels)

In [4]:
path_load = './data_source_sub'


##train
X_train = np.load(os.path.join(path_load, 'Y_train.npy'))
X_train = normalize_data_z_score(X_train)
X_train = filter_signal(X_train, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_train = pd.read_csv(os.path.join(path_load, 'Z_train.csv'))
y_train = y_train.drop(columns=['ecg_id'])
y_train = y_train.to_numpy(dtype=np.float32)

z_train = pd.read_csv(os.path.join(path_load, 'T_train.csv'))
z_train = z_train.drop(columns=['ecg_id'])
z_train = z_train.to_numpy(dtype=np.float32)

##test
X_test = np.load(os.path.join(path_load, 'Y_test.npy'))
X_test = normalize_data_z_score(X_test)
X_test = filter_signal(X_test, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_test = pd.read_csv(os.path.join(path_load, 'Z_test.csv'))
y_test = y_test.drop(columns=['ecg_id'])
y_test = y_test.to_numpy(dtype=np.float32)

z_test = pd.read_csv(os.path.join(path_load, 'T_test.csv'))
z_test = z_test.drop(columns=['ecg_id'])
z_test = z_test.to_numpy(dtype=np.float32)

##validation
X_val = np.load(os.path.join(path_load, 'Y_valid.npy'))
X_val = normalize_data_z_score(X_val)
X_val = filter_signal(X_val, axis=1, fs=100, lowcut=0.5, highcut=45.0)

y_val = pd.read_csv(os.path.join(path_load, 'Z_valid.csv'))
y_val = y_val.drop(columns=['ecg_id'])
y_val = y_val.to_numpy(dtype=np.float32)

z_val = pd.read_csv(os.path.join(path_load, 'T_valid.csv'))
z_val = z_val.drop(columns=['ecg_id'])
z_val = z_val.to_numpy(dtype=np.float32)

table = [
    ["X_train", X_train.shape],
    ["y_train", y_train.shape],
    ["z_train", z_train.shape],
    ["X_test", X_test.shape],
    ["y_test", y_test.shape],
    ["z_test", z_test.shape],
    ["X_val", X_val.shape],
    ["y_val", y_val.shape],
    ["z_val", z_val.shape],
]

print(tabulate(table, headers=["Dataset", "Shape"], tablefmt="grid"))

+-----------+-------------------+
| Dataset   | Shape             |
+===========+===================+
| X_train   | (17418, 1000, 12) |
+-----------+-------------------+
| y_train   | (17418, 5)        |
+-----------+-------------------+
| z_train   | (17418, 23)       |
+-----------+-------------------+
| X_test    | (2198, 1000, 12)  |
+-----------+-------------------+
| y_test    | (2198, 5)         |
+-----------+-------------------+
| z_test    | (2198, 23)        |
+-----------+-------------------+
| X_val     | (2183, 1000, 12)  |
+-----------+-------------------+
| y_val     | (2183, 5)         |
+-----------+-------------------+
| z_val     | (2183, 23)        |
+-----------+-------------------+


In [5]:
columns_of_csv = pd.read_csv(os.path.join(path_load, 'Z_train.csv'))
label_map = {label: i for i, label in enumerate(columns_of_csv.columns[1:])}
print(label_map)

columns_of_csv = pd.read_csv(os.path.join(path_load, 'T_train.csv'))
label_map_sub = {label: i for i, label in enumerate(columns_of_csv.columns[1:])}
print(label_map_sub)

superclass_dict = {
    'NORM': ['NORM'],
    'CD': ['LAFB/LPFB', 'IRBBB', 'ILBBB', 'CLBBB', 'CRBBB', '_AVB', 'IVCB'],
    'HYP': ['LVH', 'RVH', 'LAO/LAE', 'RAO/RAE', 'SEHYP'],
    'MI': ['AMI', 'IMI', 'LMI', 'PMI', 'ISCA', 'ISCI'],
    'STTC': ['ISC_', 'STTC', 'NST_']
}


{'NORM': 0, 'CD': 1, 'HYP': 2, 'MI': 3, 'STTC': 4}
{'NORM': 0, 'LAFB/LPFB': 1, 'IRBBB': 2, 'ILBBB': 3, 'CLBBB': 4, 'CRBBB': 5, '_AVB': 6, 'IVCB': 7, 'WPW': 8, 'LVH': 9, 'RVH': 10, 'LAO/LAE': 11, 'RAO/RAE': 12, 'SEHYP': 13, 'AMI': 14, 'IMI': 15, 'LMI': 16, 'PMI': 17, 'ISCA': 18, 'ISCI': 19, 'ISC_': 20, 'STTC': 21, 'NST_': 22}


In [6]:
# Identify the subclasses with low F1-Score
low_f1_subclasses = ["IVCB", "RVH", "SEHYP", "PMI"]

# Function to remove samples based on the low F1-Score subclasses
def remove_low_f1_samples(X, y, z, low_f1_subclasses, label_map_sub):
    # Find the indices to remove based on the low F1-Score subclasses
    to_remove_indices = []
    
    # Iterate over each sample
    for i in range(z.shape[0]):  # Iterate through the samples
        # Check if the sample has a low F1-Score subclass
        sample_labels = [label_map_sub[label] for label in low_f1_subclasses if label in label_map_sub]
        
        if any(z[i, label] == 1 for label in sample_labels):
            to_remove_indices.append(i)
    updated_label_map_sub = {k: v for k, v in label_map_sub.items() if k not in low_f1_subclasses}
    low_index = [label_map_sub[i] for i in low_f1_subclasses]
    # Remove the identified indices from X, y, and z
    X_filtered = np.delete(X, to_remove_indices, axis=0)
    y_filtered = np.delete(y, to_remove_indices, axis=0)
    z_filtered = np.delete(z, to_remove_indices, axis=0)
    z_filtered = np.delete(z_filtered, low_index, axis=1)
    return X_filtered, y_filtered, z_filtered, updated_label_map_sub

# Example usage: Remove samples where the F1-Score of the subclasses in `low_f1_subclasses` is below the threshold

# Assuming you already have X_train, y_train, z_train, X_test, y_test, z_test, etc.
X_train_filtered, y_train_filtered, z_train_filtered, label_map_sub_filtered  = remove_low_f1_samples(X_train, y_train, z_train, low_f1_subclasses, label_map_sub)
X_test_filtered, y_test_filtered, z_test_filtered, _ = remove_low_f1_samples(X_test, y_test, z_test, low_f1_subclasses, label_map_sub)
X_val_filtered, y_val_filtered, z_val_filtered, _ = remove_low_f1_samples(X_val, y_val, z_val, low_f1_subclasses, label_map_sub)

# The filtered datasets are now updated
print("Filtered datasets:")
label_map_sub = label_map_sub_filtered
label_map_sub = {label: idx for idx, label in enumerate(label_map_sub.keys())}

Filtered datasets:


In [7]:
class_counts = np.sum(z_train_filtered, axis=0)
print(class_counts)
min_class_idx = np.argmin(class_counts)  # Index of the class with the lowest occurrence
print(min_class_idx)

[7596. 1392.  848.   62.  427.  387.  651.   64. 1687.  337.   63. 2452.
 2607.  160.  750.  315. 1006. 1790.  614.]
3


In [8]:
import numpy as np

def replicate_dataset(x, y, z, threshold=2000):
    class_occurrences = np.sum(z, axis=0)
    max_occurrence = np.max(class_occurrences)
    min_occurrence = np.min(class_occurrences)
    print("------------Replicate-Dataset--------------------------------------------------")
    print(f"Max occurrence: {max_occurrence}, Min occurrence: {min_occurrence}")
    print(f"X_train shape: {x.shape}, y_train shape: {z.shape}  (before replication)")
    class_counts = np.sum(z, axis=0)
    print(f"Number of class {class_counts}")
    for class_idx, count in enumerate(class_occurrences):
        if count < threshold:
            count = int(count)
            print(f"Class {class_idx} has {count} occurrences, below threshold of {threshold}")
            
            # Calculate how many samples to replicate for this class
            num_replicates = int((threshold // count))  # Calculate how many times to replicate
            print(f"Replication factor for class {class_idx}: {num_replicates}")
            if num_replicates > 1:
                print(f"Replicating class {class_idx}: {num_replicates} times")
                
                # Get indices of samples belonging to the current class
                indices_of_class = np.where(z[:, class_idx] == 1.0)[0]
                # Randomly select the indices of the class and replicate them
                replicated_indices = np.random.choice(indices_of_class, size=(num_replicates - 1) * count, replace=True)
                
                # Append the replicated samples to the dataset
                x_replicated = x[replicated_indices]
                y_replicated = y[replicated_indices]
                z_replicated = z[replicated_indices]
                
                x = np.concatenate((x, x_replicated), axis=0)
                y = np.concatenate((y, y_replicated), axis=0)
                z = np.concatenate((z, z_replicated), axis=0)
    print(f"X_train shape: {x.shape}, y_train shape: {y.shape}  (after replication)")
    class_counts = np.sum(z, axis=0)
    print(f"Number of class {class_counts.shape}")
    print("------------------------------------------------------------------------------")
    return x, y, z


In [9]:
x_r, y_r, z_r = replicate_dataset(X_val_filtered, y_val_filtered, z_val_filtered, threshold=500)

------------Replicate-Dataset--------------------------------------------------
Max occurrence: 955.0, Min occurrence: 7.0
X_train shape: (2166, 1000, 12), y_train shape: (2166, 19)  (before replication)
Number of class [955. 179. 102.   7.  54.  52.  81.   7. 208.  43.  10. 305. 324.  20.
  92.  39. 125. 224.  75.]
Class 1 has 179 occurrences, below threshold of 500
Replication factor for class 1: 2
Replicating class 1: 2 times
Class 2 has 102 occurrences, below threshold of 500
Replication factor for class 2: 4
Replicating class 2: 4 times
Class 3 has 7 occurrences, below threshold of 500
Replication factor for class 3: 71
Replicating class 3: 71 times
Class 4 has 54 occurrences, below threshold of 500
Replication factor for class 4: 9
Replicating class 4: 9 times
Class 5 has 52 occurrences, below threshold of 500
Replication factor for class 5: 9
Replicating class 5: 9 times
Class 6 has 81 occurrences, below threshold of 500
Replication factor for class 6: 6
Replicating class 6: 6 t

In [10]:
import os
path = './data_source_replicate'
np.save(os.path.join(path, 'x_val.npy'), x_r)
np.save(os.path.join(path, 'y_val.npy'), y_r)
np.save(os.path.join(path, 'z_val.npy'), z_r)

In [11]:
# import numpy as np

# # Example y_train with shape (10000, 5)
# y_train = np.random.randint(0, 2, size=(10000, 5))  # Random binary labels for illustration

# # Define a threshold for the minimum class occurrence
# threshold = 5000

# # Count how many times each class occurs (number of 1s in each column)
# class_occurrences = np.sum(y_train, axis=0)

# # Iterate over all classes and replicate those with counts below the threshold
# for class_idx, count in enumerate(class_occurrences):
#     if count < threshold:
#         print(f"Class {class_idx} has {count} samples, which is lower than the threshold ({threshold}).")
        
#         # Get the indices where the class has a '1'
#         indices_of_class = np.where(y_train[:, class_idx] == 1)[0]

#         # Calculate how many samples you need to replicate to balance the dataset
#         max_class_count = np.max(class_occurrences)  # The class with the highest occurrence
#         replication_count = max_class_count - count  # Number of times to replicate this class

#         # Replicate the indices to balance the dataset
#         replicated_indices = np.random.choice(indices_of_class, size=replication_count, replace=True)

#         # Combine the original indices with the replicated indices
#         balanced_indices = np.concatenate([indices_of_class, replicated_indices])

#         # Shuffle the indices to avoid any ordering bias
#         np.random.shuffle(balanced_indices)

#         # Create the balanced y_train using the new indices
#         y_train[balanced_indices] = y_train[balanced_indices]

# # Print the balanced class occurrences
# balanced_class_occurrences = np.sum(y_train, axis=0)
# print("Balanced class occurrences:", balanced_class_occurrences)


In [12]:
# x_train = torch.from_numpy(X_train_filtered).float()  # float tensor for input
# labels_train = torch.from_numpy(y_train_filtered).float()  # float tensor for labels
# sub_labels_train = torch.from_numpy(z_train_filtered).float()

# x_val = torch.from_numpy(X_val_filtered).float()
# labels_val = torch.from_numpy(y_val_filtered).float()
# sub_labels_val = torch.from_numpy(z_val_filtered).float()

# x_test = torch.from_numpy(X_test_filtered).float()
# lables_test = torch.from_numpy(y_test_filtered).float()
# sub_labels_test = torch.from_numpy(z_test_filtered).float()

# batch_size = 32

# train_dataset = TensorDataset(x_train, labels_train, sub_labels_train)
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# val_dataset = TensorDataset(x_val, labels_val, sub_labels_val)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# test = TensorDataset(x_test, lables_test, sub_labels_test)
# test_loader = DataLoader(test, batch_size=batch_size, shuffle=False)